In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("GPU devices:", tf.config.list_physical_devices('GPU'))

dataset_dir = "/kaggle/input/datasets/alxmamaev/flowers-recognition/flowers"  # path to flowers dataset folder

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% validation
)

valid_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen_resnet = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

valid_gen_resnet = valid_datagen.flow_from_directory(
    dataset_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)
train_gen_googlenet = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(299,299),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

valid_gen_googlenet = valid_datagen.flow_from_directory(
    dataset_dir,
    target_size=(299,299),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)
def build_model(model_type="resnet", num_classes=5):
    if model_type=="resnet":
        base = tf.keras.applications.ResNet50(weights='imagenet',
                                              include_top=False,
                                              input_shape=(224,224,3))
    elif model_type=="googlenet":
        base = tf.keras.applications.InceptionV3(weights='imagenet',
                                                 include_top=False,
                                                 input_shape=(299,299,3))
    else:
        raise ValueError("Choose 'resnet' or 'googlenet'")
    
    base.trainable = False  
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    output = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base.input, outputs=output)
    model.compile(optimizer=Adam(1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

with tf.device('/GPU:0'):
    resnet_model = build_model("resnet", num_classes=5)
    resnet_history = resnet_model.fit(
        train_gen_resnet,
        validation_data=valid_gen_resnet,
        epochs=5,
        steps_per_epoch=train_gen_resnet.samples//32,
        validation_steps=valid_gen_resnet.samples//32
    )

with tf.device('/GPU:0'):
    googlenet_model = build_model("googlenet", num_classes=5)
    googlenet_history = googlenet_model.fit(
        train_gen_googlenet,
        validation_data=valid_gen_googlenet,
        epochs=5,
        steps_per_epoch=train_gen_googlenet.samples//32,
        validation_steps=valid_gen_googlenet.samples//32
    )
resnet_val_acc = resnet_history.history['val_accuracy'][-1]
googlenet_val_acc = googlenet_history.history['val_accuracy'][-1]

print(f"ResNet50 Validation Accuracy: {resnet_val_acc*100:.2f}%")
print(f"GoogleNet (InceptionV3) Validation Accuracy: {googlenet_val_acc*100:.2f}%")

Num GPUs Available: 2
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Found 3457 images belonging to 5 classes.
Found 860 images belonging to 5 classes.
Found 3457 images belonging to 5 classes.
Found 860 images belonging to 5 classes.


I0000 00:00:1775473142.509308      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775473142.515343      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


...

  2/108 ━━━━━━━━━━━━━━━━━━━━ 8s 79ms/step - accuracy: 0.3047 - loss: 1.9504  

I0000 00:00:1775473160.406108     159 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


108/108 ━━━━━━━━━━━━━━━━━━━━ 99s 798ms/step - accuracy: 0.2801 - loss: 1.6304 - val_accuracy: 0.3053 - val_loss: 1.5221
Epoch 2/5
  1/108 ━━━━━━━━━━━━━━━━━━━━ 9s 90ms/step - accuracy: 0.3125 - loss: 1.5705

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.3125 - loss: 1.5705 - val_accuracy: 0.3137 - val_loss: 1.5175
Epoch 3/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 44s 410ms/step - accuracy: 0.3487 - loss: 1.5156 - val_accuracy: 0.3245 - val_loss: 1.5509
Epoch 4/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.3750 - loss: 1.4967 - val_accuracy: 0.3558 - val_loss: 1.5488
Epoch 5/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 45s 412ms/step - accuracy: 0.3430 - loss: 1.5197 - val_accuracy: 0.3978 - val_loss: 1.4672
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 108s 830ms/step - accuracy: 0.6269 - loss: 1.0284 - val_accuracy: 0.8353 - val_loss: 0.4794
Epoch 2/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.7188 - loss: 0.5930 - val_accuracy: 0.8353 - val_loss: 0.4868
Epoch 3/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 76s 699ms/step - accuracy: 0.8419 - loss: 0.4672 - val_accuracy: 0.8438 - val_loss: 0.4181
Epoch 4/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - 

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Verify GPU
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("GPU devices:", tf.config.list_physical_devices('GPU'))

dataset_dir = "/kaggle/input/datasets/alxmamaev/flowers-recognition/flowers"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

valid_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

valid_gen = valid_datagen.flow_from_directory(
    dataset_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

base = tf.keras.applications.ResNet50(weights='imagenet',
                                      include_top=False,
                                      input_shape=(224,224,3))

base.trainable = False

x = base.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
output = Dense(5, activation='softmax')(x)

model = Model(inputs=base.input, outputs=output)
model.compile(optimizer=Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

with tf.device('/GPU:0'):
    history_initial = model.fit(
        train_gen,
        validation_data=valid_gen,
        epochs=5,
        steps_per_epoch=train_gen.samples//32,
        validation_steps=valid_gen.samples//32
    )

base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False
model.compile(optimizer=Adam(1e-5), 
              loss='categorical_crossentropy',
              metrics=['accuracy'])

with tf.device('/GPU:0'):
    history_finetune = model.fit(
        train_gen,
        validation_data=valid_gen,
        epochs=5, 
        steps_per_epoch=train_gen.samples//32,
        validation_steps=valid_gen.samples//32
    )

val_acc = history_finetune.history['val_accuracy'][-1]
print(f"ResNet50 Fine-tuned Validation Accuracy: {val_acc*100:.2f}%")

Num GPUs Available: 2
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Found 3457 images belonging to 5 classes.
Found 860 images belonging to 5 classes.
Epoch 1/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 59s 460ms/step - accuracy: 0.3276 - loss: 1.5693 - val_accuracy: 0.3618 - val_loss: 1.5073
Epoch 2/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.2812 - loss: 1.5531 - val_accuracy: 0.3534 - val_loss: 1.5109
Epoch 3/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 43s 402ms/step - accuracy: 0.3326 - loss: 1.5137 - val_accuracy: 0.3642 - val_loss: 1.4897
Epoch 4/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.2812 - loss: 1.6555 - val_accuracy: 0.3594 - val_loss: 1.4920
Epoch 5/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 43s 400ms/step - accuracy: 0.3537 - loss: 1.4964 - val_accuracy: 0.3161 - val_loss: 1.5355
Epoch 1/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 79s 512ms/step - accuracy: 0.3375 - loss: 1.7585 - val_accuracy: